# xAAEnet Model

> xAAEnet is a fastai adversarial autoencoder composed of an encoder, a latent space, a discriminator, a classifier, and a decoder.

In [ ]:
#| default_exp model_aae

This module provides the main xAAEnet architecture. It contains the `AAE` model, its U-Net decoder with dropout on skip connections, and the loss functions used during the different training phases.

## Overview
![AAE architecture diagram](images/schema_bloc_AAE.png)
The model follows three main steps:

1. encode the image with a ResNet34 backbone;
2. project the features into a latent space `z` constrained by a discriminator;
3. reconstruct the image with a U-Net decoder and produce a class prediction from the latent space.

In [ ]:
#| export
#| hide
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch import Tensor
from fastai.vision.all import *
from fastai.callback.hook import *
from fastai.torch_core import TensorBase
from pytorch_msssim import ms_ssim

## `AAE` Class

The `AAE` class brings together the main xAAEnet components:

- a truncated ResNet34 encoder;
- a latent bottleneck `z` with dimension `encoding_dims`;
- a linear classification head;
- a discriminator that constrains the latent space `z` to a Gaussian distribution;
- a U-Net decoder that reconstructs the image.

In [ ]:
#| export
class AAE(nn.Module):
    """Adversarial autoencoder used by xAAEnet.

    The model encodes an image into a latent vector `zi`, predicts class logits
    from this latent space, regularizes the latent distribution with an
    adversarial discriminator, and reconstructs the input image with a U-Net
    decoder.

    Parameters
    ----------
    input_size : int, default=256
        Spatial size of the square input images.
    input_channels : int, default=3
        Number of input image channels.
    encoding_dims : int, default=128
        Dimension of the latent representation `zi`.
    classes : int, default=2
        Number of output classes for the classifier head.
    gen_train : bool, default=True
        Whether the adversarial step trains the generator side of the AAE.
    skip_dropout : float, default=1
        Dropout probability applied on decoder skip connections.
    """

    def __init__(
        self,
        input_size=256,
        input_channels=3,
        encoding_dims=128,
        classes=2, 
        gen_train=True,
        skip_dropout=1 # Replaces skip_weight with skip_dropout
    ):
        super(AAE, self).__init__()

        self.gen_train = gen_train
        self.count_acc = 1
        self.classes = classes
        
        # A. Base encoder
        encoder_base = nn.Sequential(*list(resnet34(weights=ResNet34_Weights.DEFAULT).children())[:-2])
        
        # Instantiate the new U-Net architecture.
        self.unet = DynamicUnetSkipDropout(
            encoder=encoder_base, 
            n_out=input_channels, 
            img_size=(input_size, input_size), 
            skip_dropout=skip_dropout,
            last_cross=False # Crucial: disables the source-image residual connection at the output
        )
        
        # B. xAI bottleneck
        flat_features = 512 * 8 * 8
        self.flatten = nn.Flatten()
        
        self.fc_encode = nn.Linear(flat_features, encoding_dims)
        self.bn_lin = nn.BatchNorm1d(num_features=encoding_dims)
        self.decoder_fc = nn.Linear(encoding_dims, flat_features)

        # C. Network heads
        self.dropout = nn.Dropout(p=0.2)
        self.linear = nn.Linear(encoding_dims, self.classes, bias=True) 

        self.fc_crit1 = nn.Linear(encoding_dims, 64)
        self.fc_crit2 = nn.Linear(64, 16)
        self.fc_crit3 = nn.Linear(16, 1)

        self.bn_crit1 = nn.BatchNorm1d(num_features=64)
        self.bn_crit2 = nn.BatchNorm1d(num_features=16)

    def latent_gan(self, zi: Tensor) -> Tensor:
        x = F.leaky_relu(self.bn_crit1(self.fc_crit1(zi)), negative_slope=0.2)
        x = F.leaky_relu(self.bn_crit2(self.fc_crit2(x)),  negative_slope=0.2)
        x = torch.sigmoid(self.fc_crit3(x)) 
        return x
    
    def denoising_ae_loss_func(self, clean_xb, pred, yb):
        # pred and yb are ignored because this function is dedicated to denoising AE pretraining.
        # fastai expects this loss-function signature even if not all arguments are used.
        alpha = 0.84
        l1_loss = F.l1_loss(self.decoder_output, clean_xb)
        ms_ssim_val = ms_ssim(self.decoder_output, clean_xb, data_range=1.0, size_average=True)
        msssim_loss = 1.0 - ms_ssim_val
        self.recons_loss = alpha * msssim_loss + (1.0 - alpha) * l1_loss
        return self.recons_loss 

    def classif_loss_func(self, output, target, RECONS_WEIGHT, CLASS_WEIGHT, **kwargs):
        alpha = 0.84
        # Use self.input_image because there is no corruption here.
        l1_loss = F.l1_loss(self.decoder_output, self.input_image)
        ms_ssim_val = ms_ssim(self.decoder_output, self.input_image, data_range=1.0, size_average=True)
        msssim_loss = 1.0 - ms_ssim_val
        self.recons_loss = alpha * msssim_loss + (1.0 - alpha) * l1_loss

        # Store the attribute for LossAttrMetric.
        self.classif_loss = F.cross_entropy(output, target, **kwargs)
        
        return CLASS_WEIGHT * F.cross_entropy(output, target, **kwargs) + RECONS_WEIGHT * self.recons_loss
    
    def aae_loss_func(self, output, target, RECONS_WEIGHT, CLASS_WEIGHT, ADV_WEIGHT, **kwargs):
        adversarial_loss = nn.BCELoss()
        alpha = 0.84
        
        # Use self.input_image here too.
        l1_loss = F.l1_loss(self.decoder_output, self.input_image)
        ms_ssim_val = ms_ssim(self.decoder_output, self.input_image, data_range=1.0, size_average=True)
        msssim_loss = 1.0 - ms_ssim_val
        self.recons_loss = alpha * msssim_loss + (1.0 - alpha) * l1_loss

        if self.gen_train: 
            valid = torch.ones_like(self.gan_fake, requires_grad=False).detach()
            self.adv_loss = adversarial_loss(self.gan_fake, valid)
            self.crit_loss = 0
        else:
            valid = torch.ones_like(self.gan_real, requires_grad=False).detach()
            fake = torch.zeros_like(self.gan_fake, requires_grad=False).detach()
            self.real_loss = adversarial_loss(self.gan_real, valid)
            self.fake_loss = adversarial_loss(self.gan_fake, fake)
            self.adv_loss = 0.6 * self.real_loss + 0.4 * self.fake_loss
            self.crit_loss = self.adv_loss

        self.classif_loss = F.cross_entropy(output, target, **kwargs)

        loss = ADV_WEIGHT * self.adv_loss + RECONS_WEIGHT * self.recons_loss + CLASS_WEIGHT * self.classif_loss
            
        return loss

    def forward(self, x):
        self.input_image = x

        # =========================================================
        # STEP 1: ENCODER
        # =========================================================
        # Silently triggers skip-connection storage (self.unet.sfs).
        feats = self.unet.layers[0](x)

        # =========================================================
        # STEP 2: AAE BOTTLENECK
        # =========================================================
        flat = self.flatten(feats)
        self.zi = F.leaky_relu(self.bn_lin(self.fc_encode(flat)), negative_slope=0.2)
        
        labels = self.linear(self.zi)
        
        self.gan_fake = self.latent_gan(self.zi)
        z_random = torch.randn_like(self.zi)
        self.gan_real = self.latent_gan(z_random)

        # =========================================================
        # STEP 3: DECODER
        # =========================================================
        z_spatial = F.relu(self.decoder_fc(self.zi))
        z_spatial = z_spatial.view(-1, 512, 8, 8) 
        
        out = TensorBase(z_spatial)
        
        # Keep a ghost tensor for compatibility with ResizeToOrig.
        ghost_shape = torch.zeros_like(self.input_image)
        orig_x = TensorBase(ghost_shape)
        
        # Run the decoder, including the standard fastai bottleneck and UnetBlocks.
        for layer in self.unet.layers[1:]:
            out.orig = orig_x
            nres = layer(out)
            
            # Clean up VRAM.
            out.orig = None
            if hasattr(nres, 'orig'):
                nres.orig = None
                
            out = nres
            
        self.decoder_output = out

        return labels

## U-Net Decoder

The decoder reconstructs the image from the latent vector projected into a spatial representation. It follows the idea behind `DynamicUnet`: encoder features are retrieved by hooks, then injected into the decoder through skip connections.

The variant used here adds `Dropout2d` on the skip connections to limit the decoder's dependence on details passed directly by the encoder. This makes it possible to visualize the features that remain after compression by the encoder.

In [ ]:
#| export
#| hide
def _get_sz_change_idxs(sizes):
    """Identify indices where the encoder feature-map spatial size changes."""
    feature_szs = []
    for size in sizes:
        # Walk through each size tuple (C, H, W) and keep the spatial dimension.
        W = size[-1]
        feature_szs.append(W)
    return list(np.where(np.array(feature_szs[:-1]) != np.array(feature_szs[1:]))[0])

class UnetBlock(Module):
    @delegates(ConvLayer.__init__)
    def __init__(self, up_in_c, x_in_c, hook,
                 final_div=True, blur=False,
                 act_cls=defaults.activation,
                 self_attention=False,
                 init=nn.init.kaiming_normal_,
                 norm_type=None,
                 skip_dropout=0.0,
                 **kwargs):

        self.hook = hook

        # Upsampling
        self.shuf = PixelShuffle_ICNR(
            up_in_c, up_in_c//2,
            blur=blur,
            act_cls=act_cls,
            norm_type=norm_type
        )

        self.bn = BatchNorm(x_in_c)

        # 2D spatial dropout applied only to the skip connection.
        self.skip_dropout = nn.Dropout2d(skip_dropout) if skip_dropout > 0 else None

        ni = up_in_c//2 + x_in_c
        nf = ni if final_div else ni//2

        self.conv1 = ConvLayer(ni, nf, act_cls=act_cls, norm_type=norm_type, **kwargs)
        self.conv2 = ConvLayer(
            nf, nf,
            act_cls=act_cls,
            norm_type=norm_type,
            xtra=SelfAttention(nf) if self_attention else None,
            **kwargs
        )

        self.relu = act_cls()
        apply_init(nn.Sequential(self.conv1, self.conv2), init)

    def forward(self, up_in):
        s = self.hook.stored  # Retrieve encoder features.
        up_out = self.shuf(up_in)

        # Align spatial dimensions if needed.
        if s.shape[-2:] != up_out.shape[-2:]:
            up_out = F.interpolate(up_out, s.shape[-2:], mode='nearest')

        s = self.bn(s)

        # Apply dropout to the skip-connection spatial channels.
        if self.skip_dropout is not None:
            s = self.skip_dropout(s)

        cat_x = self.relu(torch.cat([up_out, s], dim=1))
        return self.conv2(self.conv1(cat_x))

class ResizeToOrig(Module):
    """Resize the tensor to the original image size stored in the .orig attribute."""
    def __init__(self, mode='nearest'):
        self.mode = mode

    def forward(self, x):
        if x.orig.shape[-2:] != x.shape[-2:]:
            x = F.interpolate(x, x.orig.shape[-2:], mode=self.mode)
        return x

class DynamicUnetSkipDropout(SequentialEx):
    """Dynamic U-Net with Dropout2d on skip connections."""
    def __init__(self, encoder, n_out, img_size,
                 blur=False, blur_final=True,
                 self_attention=False,
                 y_range=None,
                 last_cross=True,
                 bottle=False,
                 act_cls=defaults.activation,
                 init=nn.init.kaiming_normal_,
                 norm_type=None,
                 skip_dropout=0.0,
                 **kwargs):

        imsize = img_size
        sizes = model_sizes(encoder, size=imsize)
        sz_chg_idxs = list(reversed(_get_sz_change_idxs(sizes)))

        # Initialize hooks on the encoder.
        self.sfs = hook_outputs([encoder[i] for i in sz_chg_idxs], detach=False)

        x = dummy_eval(encoder, imsize).detach()
        ni = sizes[-1][1]

        # Standard U-Net bottleneck.
        middle_conv = nn.Sequential(
            ConvLayer(ni, ni*2, act_cls=act_cls, norm_type=norm_type, **kwargs),
            ConvLayer(ni*2, ni, act_cls=act_cls, norm_type=norm_type, **kwargs)
        ).eval()

        x = middle_conv(x)
        layers = [encoder, BatchNorm(ni), nn.ReLU(), middle_conv]

        # Build the decoder.
        for i, idx in enumerate(sz_chg_idxs):
            not_final = i != len(sz_chg_idxs)-1

            up_in_c = int(x.shape[1])
            x_in_c  = int(sizes[idx][1])

            do_blur = blur and (not_final or blur_final)
            sa = self_attention and (i == len(sz_chg_idxs)-3)

            unet_block = UnetBlock(
                up_in_c, x_in_c, self.sfs[i],
                final_div=not_final,
                blur=do_blur,
                self_attention=sa,
                act_cls=act_cls,
                init=init,
                norm_type=norm_type,
                skip_dropout=skip_dropout,
                **kwargs
            ).eval()

            layers.append(unet_block)
            x = unet_block(x)

        ni = x.shape[1]

        if imsize != sizes[0][-2:]:
            layers.append(PixelShuffle_ICNR(ni, act_cls=act_cls, norm_type=norm_type))

        layers.append(ResizeToOrig())

        if last_cross:
            layers.append(MergeLayer(dense=True))
            ni += in_channels(encoder)
            layers.append(
                ResBlock(
                    1, ni, ni//2 if bottle else ni,
                    act_cls=act_cls, norm_type=norm_type, **kwargs
                )
            )

        layers += [ConvLayer(ni, n_out, ks=1, act_cls=None, norm_type=norm_type, **kwargs)]
        apply_init(nn.Sequential(layers[3], layers[-2]), init)

        if y_range is not None:
            layers.append(SigmoidRange(*y_range))

        layers.append(ToTensorBase())
        super().__init__(*layers)

    def __del__(self):
        if hasattr(self, "sfs"):
            self.sfs.remove()

## Attributes Produced During `forward`

After a forward pass, the model stores several attributes that are later used by losses and visualizations:

- `input_image`: original input image;
- `zi`: latent vector produced by the encoder;
- `gan_fake`: discriminator score on the encoded latent vector;
- `gan_real`: discriminator score on a simulated Gaussian latent vector;
- `decoder_output`: image reconstruction produced by the decoder.

These attributes explain why the loss functions are class methods: they depend on intermediate values computed in `forward`.

## Minimal Example

Instantiating the model may download the pretrained ResNet34 weights. To avoid slowing down nbdev tests, the example below is shown but not executed.

## Device Selection

`default_device` returns the best accelerator available on the machine: CUDA, Apple Silicon MPS, then CPU as a fallback. The model does not force this choice automatically: the user remains free to move the model and batches to the desired device.

In [ ]:
#| export
def default_device():
    """Return the best available PyTorch device: CUDA, MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

`default_device` is useful in training scripts to move the model and batches to the right accelerator.

In [ ]:
#| eval: false
device = default_device()
model = AAE().to(device)

In [ ]:
#| eval: false
model = AAE(input_size=256, input_channels=3, encoding_dims=128, classes=2)
model

AAE(
  (unet): DynamicUnetSkipDropout(
    (layers): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (4): Sequential(
          (0): BasicBlock(
            (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu): ReLU(inplace=True)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (1): BasicBlock(
            (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 